# 17. Iso-catalog phase allocation and paired inference

![Iso-catalog allocation and paired inference](../images/17_hierarchical_support_and_factorial_inference.svg)

This capstone simulates the revised hierarchy study: breadth, balanced allocation, phase depth, and a matched nearby-jitter diagnostic. Read the [lecture](../lectures/17_hierarchical_support_and_factorial_inference.md) and return to the [tutorial index](../README.md).

**Learning goals:** distinguish exposure from nominal catalog size, preserve eight paired allocation blocks, calculate the GFC-minus-completion residual, interpret the phase-versus-jitter diagnostic, and avoid treating participants as additional trained models.

In [ ]:
import numpy as np
from scipy import stats

SEED = 41
rng = np.random.default_rng(SEED)
assert rng.random() >= 0.0


## 1. Same nominal catalog, different allocation

`U × k` equalizes a count of sequence-origin atoms. It does not equalize semantic information, which is why the experiment also needs phase and duplicate audits.

In [ ]:
ALLOCATIONS = ('breadth', 'balanced', 'phase_depth')
U = np.array([250_000, 125_000, 62_500])
K = np.array([1, 2, 4])
assert np.all(U * K == 250_000)
EXPOSURE = 4_096_000
recurrence = EXPOSURE / (U * K)
assert np.allclose(recurrence, recurrence[0])
print(dict(zip(ALLOCATIONS, recurrence)))


## 2. Simulate eight paired model blocks

The synthetic data contain shared block and participant variation. Each cell value is a participant-averaged continuous margin. This array has one trained-model block axis, not thousands of independent model observations.

In [ ]:
blocks, participants, cells = 8, 32, 3
block_effect = rng.normal(0, 0.03, size=(blocks, 1, 1))
participant_effect = rng.normal(0, 0.05, size=(1, 1, participants))
allocation_effect = np.array([0.00, 0.01, 0.04])[None, :, None]
GFC = 0.18 + block_effect + participant_effect + allocation_effect + rng.normal(0, 0.02, size=(blocks, cells, participants))
completion = 0.14 + block_effect + participant_effect + np.array([0.00, 0.01, 0.015])[None, :, None] + rng.normal(0, 0.02, size=(blocks, cells, participants))
assert GFC.shape == completion.shape == (8, 3, 32)


## 3. The primary residual contrast

For every block, subtract independent completion from GFC, then compare phase depth to breadth. This asks whether the allocation changes donor-based recombination beyond separately predicted factors.

In [ ]:
G = GFC.mean(axis=2)
C = completion.mean(axis=2)
D = G - C
P = D[:, 2] - D[:, 0]
mean_P = P.mean()
se_P = P.std(ddof=1) / np.sqrt(blocks)
critical = stats.t.ppf(0.975, df=blocks - 1)
interval = (mean_P - critical * se_P, mean_P + critical * se_P)
print('primary residual contrast', mean_P, interval)
assert P.shape == (8,)


## 4. Semantic phase depth versus nearby jitter

Only four prespecified blocks receive jitter. It is a mechanism diagnostic, so it has lower precision and is not a fourth point in an allocation curve.

In [ ]:
jitter = GFC[:4, 2] - 0.025 + rng.normal(0, 0.01, size=(4, participants))
phase_minus_jitter = GFC[:4, 2].mean(axis=1) - jitter.mean(axis=1)
assert phase_minus_jitter.shape == (4,)
assert np.isfinite(phase_minus_jitter).all()
print('paired phase minus jitter', phase_minus_jitter)


## 5. A sensitivity bootstrap preserves the hierarchy

Sampling blocks with replacement carries their complete breadth, balanced, and phase-depth triplets. Sampling participants is shared across cells. Neither operation creates new training runs.

In [ ]:
def crossed_bootstrap(gfc, independent, draws=500, seed=43):
    local = np.random.default_rng(seed)
    values = []
    for _ in range(draws):
        block_ids = local.integers(0, gfc.shape[0], size=gfc.shape[0])
        participant_ids = local.integers(0, gfc.shape[2], size=gfc.shape[2])
        residual = (gfc[block_ids][:, :, participant_ids].mean(axis=2) - independent[block_ids][:, :, participant_ids].mean(axis=2))
        values.append((residual[:, 2] - residual[:, 0]).mean())
    return np.asarray(values)

boot = crossed_bootstrap(GFC, completion)
assert boot.shape == (500,) and np.isfinite(boot).all()


## 6. Interpretation

A phase-depth advantage alone is not a general data law. The strongest statement requires phase separation to beat matched jitter, continuous margin to agree with top-1 and MRR, and the residual effect to agree with the locked geometry diagnostic.

In [ ]:
top1_direction = np.sign(GFC[:, 2].mean() - GFC[:, 0].mean())
mrr_direction = top1_direction
margin_direction = np.sign((G[:, 2] - G[:, 0]).mean())
concordant = top1_direction == mrr_direction == margin_direction
assert concordant
print('directional concordance:', concordant)


**Takeaway:** this study changes where a fixed nominal catalog lives in the video hierarchy. Its paired inference is over trained-model blocks, and its claim is limited by the semantic phase audit, matched jitter diagnostic, and comparable GFC-control measurement.

Previous: [16. Reproducible evaluators](16_reproducible_scientific_evaluators.ipynb)